## Sequential text pre-processing

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 100)

df = pd.read_csv('./data/wyoming.csv')
df.shape

(234655, 7)

In [2]:
df.loc[0,['text']]

text    When knowledge is key and kindness matters, Niki Morrison is the right combination every time.
Name: 0, dtype: object

In [3]:
df[['text']]

,text
0,"When knowledge is key and kindness matters, Niki Morrison is the right combination every time."
1,"The entire team is outstanding! They are professional, knowledgeable, and friendly. The only cho..."
2,They provided immediate and thorough help. I had purchased a laptop there a few years ago and wh...
3,"Had some work done on a printer and they have done all they can do to fix the issue, including t..."
4,"The people, wonderful people."
...,...
234650,(Translated by Google) She has gotten her clothes\n\n(Original)\nTara ta ta ta taaa me encanta🤗
234651,(Translated by Google) Very good service 24 hours\n\n(Original)\nMuy buen servicio las 24 horas
234652,(Translated by Google) Fine\n\n(Original)\nДобре
234653,(Translated by Google) I love there Carmel strikes\n\n(Original)\nI love there Carmel frappes


### Tokenize

The Keras `Tokenizer` seems to be the most modern all-in-one solution

In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(df['text'])
X_seq = tokenizer.texts_to_sequences(df['text'])

In [ ]:
len(tokenizer.word_index) # Maximum vocabulary size (may useful for setting LSTM model vocab_size)

53667

### Verify output

In [5]:
X_seq[0]

[64, 1225, 9, 1790, 3, 3082, 5095, 1, 1, 9, 2, 153, 2939, 163, 49]

In [6]:
[tokenizer.index_word[i] for i in X_seq[0]]

['when',
 'knowledge',
 'is',
 'key',
 'and',
 'kindness',
 'matters',
 '<OOV>',
 '<OOV>',
 'is',
 'the',
 'right',
 'combination',
 'every',
 'time']

### Pad sequences

Make observations of identical size.

Note: Padding increases data size, so saved files take up more space. However, train-test split on a variable-length list object is a hassle (Alteratively, add text pre-processing to model file without saving to disk.)

Note: There is some potential for "leakage" between training and test data if padding before splitting.

In [7]:
lengths = [len(seq) for seq in X_seq]
print(f'Max length: {np.max(lengths)}')
print(f'Average length: {np.mean(lengths)}')
print(f'Quantiles: {np.quantile(lengths, [0.01, 0.10, 0.25, 0.5, 0.75, 0.90, 0.99])}')

Max length: 788
Average length: 17.56331209648207
Quantiles: [  1.   2.   5.   9.  20.  40. 125.]


In [8]:
max_length = 100
X = pad_sequences(X_seq, maxlen=max_length, padding='post', truncating='post')

In [18]:
X[0:2]

array([[  64, 1225,    9, 1790,    3, 3082, 5095,    1,    1,    9,    2,
         153, 2939,  163,   49,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0],
       [   2,  742,  888,    9,  533,   17,   23,  320,  260,    3,   27,
           2,   87,  662,   13, 2787, 1412,  623,    1,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    

### Train-test split

In [ ]:
y = df['rating'] - 1 # shold be 0-4 indexed

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [14]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((187724, 100), (187724,), (46931, 100), (46931,))

In [15]:
np.save('./data/X_train_seq.npy', X_train)
np.save('./data/y_train_seq.npy', y_train)

np.save('./data/X_test_seq.npy', X_test)
np.save('./data/y_test_seq.npy', y_test)